In [1]:
import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F

with open("names.txt", "r") as f:
    names = f.read().splitlines()

# print 5 sample names
names[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [2]:
# total number of names in dataset
len(names)

32033

In [ ]:
# Create data sets (the input n-gram and corresponding expected label)
NGRAM_SIZE = 3 # The n in n-gram

# Map characters to integer indices ('.' is the boundary/padding token)
chars = ['.'] + sorted(set(''.join(names)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

X = []
Y = []
for name in names:
    window = "." * NGRAM_SIZE
    for ch in name + ".":
        X.append([stoi[c] for c in window])
        Y.append(stoi[ch])
        #print(f"{window} -> {ch}")
        window = window[1:] + ch

vocab_size = len(chars)

X = torch.tensor(X)
Y = torch.tensor(Y)
X.shape, Y.shape, len(chars)

(torch.Size([228146, 3]), torch.Size([228146]), 27)

In [73]:
class Tanh:
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out

    def parameters(self):
        return []

class Linear:
    def __init__ (self, in_features, out_features, bias=True):
        k = 1/in_features
        upper = k ** 0.5
        lower = -(k ** 0.5)
        self.weight = (upper - lower) * torch.rand((in_features, out_features)) + lower
        self.bias = None if bias == False else (upper - lower) * torch.rand((out_features)) + lower

    def __call__ (self, x): # x is typically of shape (batchsize, in_features)
        """ output is of the shape (batchsize, out_features) """
        self.out = x @ self.weight + self.bias
        return self.out

    def parameters (self):
        # Return the leaf tensors themselves. Iterating over self.weight would
        # yield row *views*, which are not leaves, so their .grad stays None.
        return [self.weight] + ([] if self.bias is None else [self.bias])

class BatchNorm:
    def __init__(self, out_features, eps=1e-05):
        self.eps = eps
        self.gamma = torch.ones((1, out_features))
        self.beta = torch.zeros((1, out_features))

    def __call__(self, x):
        """
        Input shape: (batchsize, featuresize)
        Output shape: (batchsize, featuresize) - same shape as the input
        """
        mean = torch.mean(x, dim=0, keepdim=True) # size = 1, featuresize
        variance = torch.var(x, dim=0, keepdim=True) # size = 1, featuresize
        xhat = (x - mean) / ((variance + self.eps) ** 0.5) # size = batchsize, featuresize
        self.out = (self.gamma * xhat) + self.beta # size = batchsize, featuresize
        return self.out

    def parameters(self):
        params = []
        params.append(self.gamma)
        params.append(self.beta)
        return params


In [74]:
m = Linear(20, 30)
input = torch.randn(128, 20)
output = m(input)
print(output.size())

torch.Size([128, 30])


In [ ]:
# instantiate the net

torch.manual_seed(100)
EMBEDDING_DIMENSIONS = 2
MINIBATCH_SIZE = 32

C = torch.randn((vocab_size, EMBEDDING_DIMENSIONS)) 
layers = [Linear (EMBEDDING_DIMENSIONS * NGRAM_SIZE, 100), BatchNorm(100), Tanh(), Linear(100, vocab_size)]

for layer in layers:
    for p in layer.parameters():
        p.requires_grad = True


for i in range(200000):

    # construct a batch instead of training on all examples
    batch_indexes = torch.randint(0, X.size(0), (32,))
    
    # lookup embeddings
    one_hot = F.one_hot(X[batch_indexes], num_classes=27).float() # (32, 3, 27)
    x = one_hot @ C # (32, 3, 2)

    # first linear layer is a layer of 6 neurons
    x = x.view(-1, 6)

    # call the neural net
    for layer in layers:
        x = layer(x)
    
    # compute loss
    loss = F.cross_entropy(x, Y[batch_indexes])

    # compute gradients
    loss.backward()

    # set some learning rate
    if i <= 100000:
        lr = 0.1
    else:
        lr = 0.01

    # adjust weights based on the gradients
    for layer in layers:
        for p in layer.parameters():
            p.data += -lr * p.grad

    # reset gradients to get ready for next pass
    for layer in layers:
        for p in layer.parameters():
            p.grad = None

    if i%10000 == 0:
        print(f"{i} : {loss:.4f}")



0 : 3.4492
10000 : 2.3314
20000 : 2.6063
30000 : 2.8227
40000 : 2.5205
50000 : 2.1465
60000 : 2.7291
70000 : 2.2918
80000 : 2.4077
90000 : 2.5656
100000 : 2.4150
110000 : 2.5367
120000 : 2.7726
130000 : 2.1382
140000 : 2.3126
150000 : 2.1002
160000 : 2.2026
170000 : 2.6834
180000 : 2.4046
190000 : 2.3655


In [6]:
# sample the neural net to get name sounding things ...

g = torch.Generator().manual_seed(2147483647)
for _ in range(20):
    out = []
    context = [0] * 3
    context = torch.tensor(context)
    while True:
        one_hot = F.one_hot(context, num_classes=27).float()
        embedding = one_hot @ C
        tanh_activations = torch.tanh(embedding.view(-1, 3*EMBEDDING_DIMENSIONS) @ W1 + b1)
        logits = tanh_activations @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context = torch.cat([context[1:], torch.tensor([ix])])
        out.append(ix)
        if ix == 0:
            break
    print(''.join(itos[i] for i in out))

dex.
malomaluraile.
kayda.
malimittain.
luchk.
katha.
samiyah.
javer.
gothi.
moriella.
kin.
pred.
jen.
emia.
sadel.
tkaviyn.
hatlupihiniden.
tarlyn.
dasde.
breegh.
